# Part 3 · Authentication, data isolation and human approval

Many users share one agent and one checkpoint database. Before a route runs, the service answers two questions:
**who is calling** (the bearer token) and **is this thread theirs** (ownership). Then a paused run can be resumed
over HTTP by its owner only.

| File | Role |
| --- | --- |
| [`core/security.py`](../app/core/security.py) | `HTTPBearer` + `current_user`: token → `User`, or 401 |
| [`services/threads.py`](../app/services/threads.py) | thread records in the LangGraph store, one namespace per user |
| [`dependencies.py`](../app/dependencies.py) | `owned_thread` (404), `idle_thread` / `interrupted_thread` (409) |
| [`routers/runs.py`](../app/routers/runs.py) | `POST /threads/{thread_id}/resume` → `Command(resume=…)` |

Docs: [FastAPI security](https://fastapi.tiangolo.com/tutorial/security/first-steps/) ·
[Dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/) ·
[LangChain human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) ·
[OWASP API1:2023 BOLA](https://owasp.org/API-Security/editions/2023/en/0xa1-broken-object-level-authorization/)

In [1]:
import logging

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
load_dotenv()

from utils import PROJECT_DIR, check_server, client, short, source, stream_run  # notebooks/utils.py

check_server()

server: ok


## 1. Who is calling: `current_user`

`HTTPBearer` reads `Authorization: Bearer <token>`; `current_user` maps the token to a `User` or raises **401** with
`WWW-Authenticate: Bearer`. Every route that declares `user: CurrentUser` runs it first. The demo tokens in
`data/users.json` stand in for an identity provider (OAuth / OIDC); a real service verifies a signed token instead.

In [2]:
from fastapi import HTTPException
from fastapi.security import HTTPAuthorizationCredentials

from app.core.security import current_user

source(current_user)

async def current_user(cred: Annotated[HTTPAuthorizationCredentials | None, Depends(bearer)]) -> User:
    user = load_users().get(cred.credentials) if cred else None
    if user is None:
        raise HTTPException(401, "Missing or invalid bearer token", headers={"WWW-Authenticate": "Bearer"})
    return user

In [3]:
print(await current_user(HTTPAuthorizationCredentials(scheme="Bearer", credentials="alice-token-7f3a")))
for cred in (None, HTTPAuthorizationCredentials(scheme="Bearer", credentials="guessed-token")):
    try:
        await current_user(cred)
    except HTTPException as exc:
        print(exc.status_code, exc.detail, exc.headers)

user_id='alice' name='Alice'
401 Missing or invalid bearer token {'WWW-Authenticate': 'Bearer'}
401 Missing or invalid bearer token {'WWW-Authenticate': 'Bearer'}


## 2. Whose thread: one store namespace per user

A thread record is saved under `("users", user_id, "threads")`. `get_thread` looks an id up **in the caller's
namespace only**, so another user's thread is simply not found. Try it on an in-memory store:

In [4]:
from langgraph.store.memory import InMemoryStore

from app.services import threads

source(threads.get_thread)

async def get_thread(store: BaseStore, user_id: str, thread_id: str) -> Thread | None:
    item = await store.aget(namespace(user_id), thread_id)
    return Thread(thread_id=thread_id, **item.value) if item else None

In [5]:
store = InMemoryStore()
t = await threads.create_thread(store, "alice")
print("created for alice:", t.thread_id, "in namespace", threads.namespace("alice"))
print("alice looks it up:", await threads.get_thread(store, "alice", t.thread_id))
print("bob looks it up:  ", await threads.get_thread(store, "bob", t.thread_id))

created for alice: aa7950d26dcf in namespace ('users', 'alice', 'threads')
alice looks it up: thread_id='aa7950d26dcf' title='New brief' created_at='2026-11-01T15:30:18+00:00' status='idle'
bob looks it up:   None


`owned_thread` turns "not found" into **404**, the same answer as for an id that does not exist, so ids cannot be
probed. Because it is a dependency, it runs before the route, before an SSE stream sends its `200` and first byte.

In [6]:
from app import dependencies

source(dependencies.owned_thread)

async def owned_thread(thread_id: str, user: CurrentUser, store: Store) -> Thread:
    thread = await threads.get_thread(store, user.user_id, thread_id)
    if thread is None:  # same answer for "does not exist" and "belongs to someone else": ids are not revealed
        raise HTTPException(404, f"Thread {thread_id} not found")
    return thread

## 3. Bob tries every route to Alice's thread

Alice starts a brief on the running service; it pauses for approval. Then Bob tries the direct read **and** the
indirect routes (run, resume, list, a forged `user_id` in the body). Each must fail.

In [7]:
alice, bob = client("alice"), client("bob")
thread_id = alice.post("/threads").json()["thread_id"]
events = stream_run("alice", f"/threads/{thread_id}/runs",
                    {"message": "Brief on how long electric-car batteries last."}, quiet=True)
print("Alice's run:", events[-1]["data"]["status"], "| thread:", alice.get(f"/threads/{thread_id}").json()["status"])

Alice's run: interrupted | thread: interrupted


In [8]:
bob_thread = bob.post("/threads").json()["thread_id"]
attempts = [
    ("GET  Alice's thread and its messages", lambda: bob.get(f"/threads/{thread_id}"), 404),
    ("POST a run on Alice's thread", lambda: bob.post(f"/threads/{thread_id}/runs", json={"message": "hi"}), 404),
    ("POST resume: approve Alice's save_brief", lambda: bob.post(f"/threads/{thread_id}/resume",
                                                               json={"decision": "approve"}), 404),
    ("POST a run with user_id='alice' in the body", lambda: bob.post(f"/threads/{bob_thread}/runs",
                                                                     json={"message": "hi", "user_id": "alice"}), 422),
    ("GET  a thread id that does not exist", lambda: bob.get("/threads/000000000000"), 404),
]
print(f"{'Bob tries to':<46}{'status':>7}  result")
for label, call, expected in attempts:
    r = call()
    print(f"{label:<46}{r.status_code:>7}  {'PASS' if r.status_code == expected else 'FAIL'}  {short(r.json()['detail'], 45)}")
listed = [t["thread_id"] for t in bob.get("/threads").json()]
print(f"{'GET  /threads (his list)':<46}{200:>7}  {'PASS' if thread_id not in listed else 'FAIL'}  "
      f"{len(listed)} thread(s), none of Alice's")
print("\nAlice's thread is still waiting:", alice.get(f"/threads/{thread_id}").json()["status"])

Bob tries to                                   status  result
GET  Alice's thread and its messages              404  PASS  Thread d080c0ba7ff9 not found
POST a run on Alice's thread                      404  PASS  Thread d080c0ba7ff9 not found
POST resume: approve Alice's save_brief           404  PASS  Thread d080c0ba7ff9 not found
POST a run with user_id='alice' in the body       422  PASS  The request body or parameters are invalid.
GET  a thread id that does not exist              404  PASS  Thread 000000000000 not found
GET  /threads (his list)                          200  PASS  1 thread(s), none of Alice's

Alice's thread is still waiting: interrupted


## 4. Human approval over HTTP: resume

`resume_run` accepts only an `interrupted` thread of the caller (`InterruptedThread`, else 409) and turns
`{"decision": ...}` into `Command(resume={"decisions": [...]})`, the format `HumanInTheLoopMiddleware` expects. The
rest of the run streams on a **new** SSE response.

In [9]:
from app.routers import runs

source(runs.resume_run)

@router.post("/resume", response_class=EventSourceResponse)
async def resume_run(body: ResumeRequest, thread: InterruptedThread, user: CurrentUser, agent: Agent,
                     running: Running) -> AsyncIterable[ServerSentEvent]:
    """Answer the pending approval (approve or reject) and stream the rest of the run."""
    decision = {"type": body.decision}
    if body.decision == "reject" and body.message:
        decision["message"] = body.message  # the model reads the reason
    command = Command(resume={"decisions": [decision]})  # the format HumanInTheLoopMiddleware expects
    async for event in run_events(agent, running, user.user_id, thread.thread_id, command,
                                  get_settings().run_timeout_s):
        yield event

In [10]:
pending = alice.get(f"/threads/{thread_id}").json()["interrupt"]["action_requests"][0]
print(pending["name"], "| title:", pending["args"]["title"], "\n" + pending["args"]["content"][:300], "…\n")
events = stream_run("alice", f"/threads/{thread_id}/resume", {"decision": "approve"})

save_brief | title: How long EV batteries last 
## Findings
- A large 2024 Geotab analysis of nearly 5,000 electric vehicles found average battery degradation of about 1.8% per year; its earlier 2019 analysis found 2.3% per year. This is an average, not a prediction for every vehicle. [https://www.geotab.com/uk/press-release/2024-battery-degradat …

[metadata] run 4dece15a on thread d080c0ba7ff9
[tool_call] save_brief({"title": "How long EV batteries last", "content": "## Findings\n- A large 2024 Geotab ana…)
[step] tools
[tool_result] save_brief -> success: saved: briefs/alice/how-long-ev-batteries-last.md
[token] - A 2024 analysis of nearly 5,000 EVs found average battery degradation of about **1.8% per year**—an average, not a vehicle-specific forecast.
- Many EV battery warranties cover at least **8 years or 100,000 miles**, but terms vary.
- There’s no single expiration date: cited estimates suggest many batteries last **more than 10 years**, and potentially up to 20.

**Saved** t

In [11]:
saved = next(e["data"]["preview"] for e in events if e["event"] == "tool_result" and e["data"]["name"] == "save_brief")
path = PROJECT_DIR / "output" / saved.removeprefix("saved: ")
print("file:", path.relative_to(PROJECT_DIR), "| exists:", path.exists())

file: output/briefs/alice/how-long-ev-batteries-last.md | exists: True


`save_brief` wrote under `output/briefs/alice/`: the tool takes the user from `ToolRuntime.context.user_id`, which
the server filled from the token. Now a **reject** with a reason: the tool does not run, the model reads the reason.

In [12]:
reject_thread = alice.post("/threads").json()["thread_id"]
events = stream_run("alice", f"/threads/{reject_thread}/runs",
                    {"message": "Brief on the payback time of rooftop solar panels."}, quiet=True)
print("first run:", events[-1]["data"]["status"])
events = stream_run("alice", f"/threads/{reject_thread}/resume",
                    {"decision": "reject", "message": "Do not save it yet, I will check the sources first."})
print("save_brief ran:", any(e["event"] == "tool_result" and e["data"]["name"] == "save_brief"
                            and e["data"]["status"] == "success" for e in events))

first run: interrupted
[metadata] run 8502644b on thread dd6543516b0b
[tool_call] save_brief({"title": "Rooftop solar payback", "content": "## Findings\n- For a U.S. homeowner, a usef…)
[tool_result] save_brief -> error: User rejected the tool call for `save_brief` with reason: Do not save it yet, I …
[token] - Rooftop solar payback is often estimated at **6–15 years**; one source reports an average of about **10.8 years** for U.S. shoppers.
- Location matters: reported state estimates range from **7.39 years in California** to **20.95 years in Alabama**.
- A basic estimate is net system cost divided by annual bill savings; electricity prices, incentives, financing, and utility export credits affect the result.

**Not saved**—you asked to review the sources first.
[step] model
[done] completed in 2.2 s
save_brief ran: False
